# B1.1 · Agentic SAST

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *AI for Security*

---

**Risk.** Pattern matching floods the queue; the false-positive rate is what actually changed.

**Control.** Reachability, exploitability and cross-file reasoning on top of deterministic rules.

**This lab.** Cut the false-positive rate with reachability reasoning, and measure it.

| | |
|---|---|
| Open-source tooling | OpenGrep, Semgrep OSS, CodeQL |
| Open-weight models | GLM-4.6, Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B1.1"))

Agentic SAST is not hard because finding bugs is hard. It is hard because a tool that reports everything is indistinguishable from one that reports nothing — both get muted.

In [ ]:
from cybercommons import appsec

findings = appsec.scan_all()
print(f"{len(findings)} findings across {len(appsec.SNIPPETS)} files\n")
for f in findings:
    print(f"{f.cwe:9s} {f.name:22s} {f.file}:{f.line}")
    print(f"          {f.evidence}")

Now the part that decides whether anyone acts on them: ranking by exploitability rather than by severity label.

In [ ]:
for f in appsec.triage(findings):
    print(f"  score {f.exploitability():2d}  {f.cwe:9s} {f.file}:{f.line}")

print("\nsame findings, one marked unreachable and one not shipping:")
findings[0].reachable = False
findings[1].in_prod = False
for f in appsec.triage(findings):
    print(f"  score {f.exploitability():2d}  {f.cwe:9s} {f.file}:{f.line} "
          f"reachable={f.reachable} prod={f.in_prod}")

Note the safe snippets produce nothing. A scanner that fires on parameterised SQL has a precision problem that no amount of triage fixes downstream.

### Expect

Findings for CWE-89, CWE-78, CWE-22 and CWE-798, none of them in the `safe_*` snippets. Triage orders command injection above SQL injection above path traversal, and marking a finding unreachable drops it five points.

### Your turn

Add a snippet with a real vulnerability the regex rules miss (second-order SQL injection is a good one). That gap is why agentic review exists — and why it still needs an eval.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B1.1.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*